# HAM10000 Spurious Correlation Experiment
**ERM + Focal Loss (FL) Training with Abstention Evaluation**

Run this notebook on Kaggle with a T4 GPU.
Dataset: [HAM10000 Skin Lesion](https://www.kaggle.com/datasets/kmader/skin-lesion-analysis-toward-melanoma-detection)

In [1]:
import torch
print(torch.__version__)
print(torch.cuda.get_device_name(0))

2.10.0+cu128
Tesla T4


In [2]:
import os
import pandas as pd
import numpy as np
from PIL import Image
import random
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split, WeightedRandomSampler

import torchvision.transforms as transforms
import torchvision.models as models
from tqdm import tqdm
from sklearn.model_selection import StratifiedShuffleSplit

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [3]:
set_seed(456)

In [4]:
os.makedirs("/kaggle/working/ham10000", exist_ok=True)
os.system("kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 "
          "-p /kaggle/working/ham10000 --unzip")

DATA_ROOT = "/kaggle/working/ham10000"
META_CSV  = os.path.join(DATA_ROOT, "HAM10000_metadata.csv")
IMG_DIRS  = [
    os.path.join(DATA_ROOT, "HAM10000_images_part_1"),
    os.path.join(DATA_ROOT, "HAM10000_images_part_2"),
]

# Build image lookup across both part folders
IMG_LOOKUP = {}
for d in IMG_DIRS:
    if os.path.isdir(d):
        for fname in os.listdir(d):
            stem = os.path.splitext(fname)[0]
            IMG_LOOKUP[stem] = os.path.join(d, fname)

print(f"Found {len(IMG_LOOKUP):,} images")
# Expected: 10,015

Dataset URL: https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000
License(s): CC-BY-NC-SA-4.0


100%|██████████| 5.20G/5.20G [00:22<00:00, 243MB/s]



Found 10,015 images


In [5]:
meta = pd.read_csv(META_CSV)

# Label: melanoma vs non-melanoma
meta["y"]     = (meta["dx"] == "mel").astype(int)

# Spurious attribute: acquisition method
# histo = histopathologically confirmed (place=1)
# non-histo (consensus/follow_up/confocal) = place=0
meta["place"] = (meta["dx_type"] == "histo").astype(int)

# Group = 2*y + place → 0,1,2,3
# 0: non-melanoma + non-histo  (majority)
# 1: non-melanoma + histo      (minority)
# 2: melanoma + non-histo      (empty in HAM10000 — all melanoma is histo-confirmed)
# 3: melanoma + histo          (minority)
meta["group"] = 2 * meta["y"] + meta["place"]

print("Full dataset group distribution:")
print(meta["group"].value_counts().sort_index())
print(f"\nNote: Group 2 is empty — all melanoma cases are histopathologically confirmed.")

# Stratified 70/15/15 split by GROUP (not just label)
idx_all = np.arange(len(meta))
groups  = meta["group"].values

sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, tmp_idx = next(sss1.split(idx_all, groups))

sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_rel, test_rel = next(sss2.split(tmp_idx, groups[tmp_idx]))
val_idx  = tmp_idx[val_rel]
test_idx = tmp_idx[test_rel]

# Assign split column: 0=train, 1=val, 2=test
meta["split"] = -1
meta.loc[train_idx, "split"] = 0
meta.loc[val_idx,   "split"] = 1
meta.loc[test_idx,  "split"] = 2

META_SAVE = "/kaggle/working/ham_metadata.csv"
meta.to_csv(META_SAVE, index=False)

print(f"\nTrain: {len(train_idx):,} | Val: {len(val_idx):,} | Test: {len(test_idx):,}")
print("\nTrain group distribution:")
print(meta[meta["split"]==0]["group"].value_counts().sort_index())
print("\nVal group distribution:")
print(meta[meta["split"]==1]["group"].value_counts().sort_index())
print("\nTest group distribution:")
print(meta[meta["split"]==2]["group"].value_counts().sort_index())
print("\n✓ Verify all groups present in val/test have ≥20 samples before continuing")

Full dataset group distribution:
group
0    4675
1    4227
3    1113
Name: count, dtype: int64

Note: Group 2 is empty — all melanoma cases are histopathologically confirmed.

Train: 7,010 | Val: 1,502 | Test: 1,503

Train group distribution:
group
0    3272
1    2959
3     779
Name: count, dtype: int64

Val group distribution:
group
0    701
1    634
3    167
Name: count, dtype: int64

Test group distribution:
group
0    702
1    634
3    167
Name: count, dtype: int64

✓ Verify all groups present in val/test have ≥20 samples before continuing


In [6]:
class HAM10000Dataset(Dataset):
    def __init__(self, meta_path, img_lookup, split_name, transform=None):
        self.img_lookup = img_lookup
        self.transform  = transform

        meta = pd.read_csv(meta_path)
        split_map = {"train": 0, "val": 1, "test": 2}
        self.data = meta[meta["split"] == split_map[split_name]].reset_index(drop=True)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row    = self.data.iloc[idx]
        img_path = self.img_lookup[row["image_id"]]
        img    = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, int(row["y"]), int(row["group"]), row["image_id"]

In [7]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop((224, 224), scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_data = HAM10000Dataset(META_SAVE, IMG_LOOKUP, "train", transform=train_transform)
val_data   = HAM10000Dataset(META_SAVE, IMG_LOOKUP, "val",   transform=eval_transform)
test_data  = HAM10000Dataset(META_SAVE, IMG_LOOKUP, "test",  transform=eval_transform)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_data,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_data,  batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}")
# Expected: Train: ~7010 | Val: ~1502 | Test: ~1503

Train: 7010 | Val: 1502 | Test: 1503


In [8]:
def make_resnet50():
    m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    m.fc = nn.Linear(m.fc.in_features, 2)
    return m.to(device)

EPOCHS    = 25
criterion = nn.CrossEntropyLoss()

print(f"Training ERM on HAM10000 train set ({len(train_data)} samples) for {EPOCHS} epochs")

Training ERM on HAM10000 train set (7010 samples) for 25 epochs


In [8]:
erm_model = make_resnet50()
optimizer = optim.SGD(erm_model.parameters(), lr=1e-3, momentum=0.9, weight_decay=1e-4)

print("Starting HAM10000 ERM Training...")
best_val_wga = 0.0

for epoch in range(EPOCHS):
    erm_model.train()
    train_loss, correct, total = 0.0, 0, 0
    loop = tqdm(train_loader, desc=f"ERM Epoch [{epoch+1}/{EPOCHS}]")

    for images, targets, groups, _ in loop:
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        logits = erm_model(images)
        loss   = criterion(logits, targets)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        preds       = torch.argmax(logits, dim=1)
        correct    += (preds == targets).sum().item()
        total      += targets.size(0)
        loop.set_postfix({"loss": train_loss / (loop.n + 1), "acc": correct / total})

    # Validation
    erm_model.eval()
    group_correct = {}
    group_total   = {}
    val_correct, val_total = 0, 0

    with torch.no_grad():
        for images, targets, groups, _ in val_loader:
            images, targets = images.to(device), targets.to(device)
            logits = erm_model(images)
            preds  = torch.argmax(logits, dim=1)
            val_correct += (preds == targets).sum().item()
            val_total   += targets.size(0)
            for i in range(len(targets)):
                g = groups[i].item()
                group_total[g]   = group_total.get(g, 0) + 1
                group_correct[g] = group_correct.get(g, 0) + (preds[i] == targets[i]).item()

    group_accs = {g: group_correct[g] / (group_total[g] + 1e-8) for g in group_total}
    wga     = min(group_accs.values())
    avg_acc = val_correct / val_total

    group_str = "  ".join([f"G{g}:{group_accs[g]:.3f}" for g in sorted(group_accs)])
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Train Acc: {correct/total:.4f} "
          f"| Val Acc: {avg_acc:.4f} | Val WGA: {wga:.4f} | {group_str}")

    if wga > best_val_wga:
        best_val_wga = wga
        torch.save(erm_model.state_dict(), "/kaggle/working/ham_erm_best.pth")
        print(f"  → New best WGA: {wga:.4f} — checkpoint saved")

torch.save(erm_model.state_dict(), "/kaggle/working/ham_erm_final.pth")
print(f"\nERM training complete. Best val WGA: {best_val_wga:.4f}")

Starting HAM10000 ERM Training...


ERM Epoch [1/25]: 100%|██████████| 220/220 [01:06<00:00,  3.32it/s, loss=0.278, acc=0.891]


Epoch 01/25 | Train Acc: 0.8910 | Val Acc: 0.9115 | Val WGA: 0.3293 | G0:0.993  G1:0.975  G3:0.329
  → New best WGA: 0.3293 — checkpoint saved


ERM Epoch [2/25]: 100%|██████████| 220/220 [01:07<00:00,  3.24it/s, loss=0.215, acc=0.912]


Epoch 02/25 | Train Acc: 0.9116 | Val Acc: 0.9101 | Val WGA: 0.3772 | G0:1.000  G1:0.951  G3:0.377
  → New best WGA: 0.3772 — checkpoint saved


...

Epoch 23/25 | Train Acc: 0.9936 | Val Acc: 0.9281 | Val WGA: 0.6407 | G0:0.989  G1:0.937  G3:0.641


ERM Epoch [24/25]: 100%|██████████| 220/220 [01:10<00:00,  3.13it/s, loss=0.0164, acc=0.995]


Epoch 24/25 | Train Acc: 0.9951 | Val Acc: 0.9294 | Val WGA: 0.5389 | G0:0.996  G1:0.959  G3:0.539


ERM Epoch [25/25]: 100%|██████████| 220/220 [01:10<00:00,  3.13it/s, loss=0.00816, acc=0.998]


Epoch 25/25 | Train Acc: 0.9977 | Val Acc: 0.9374 | Val WGA: 0.6228 | G0:0.997  G1:0.954  G3:0.623

ERM training complete. Best val WGA: 0.6946


In [9]:
erm_model.load_state_dict(torch.load("/kaggle/working/ham_erm_best.pth"))
erm_model.eval()

group_correct, group_total = {}, {}
test_correct, test_total = 0, 0
logs = []

with torch.no_grad():
    for images, targets, groups, img_ids in tqdm(test_loader, desc="ERM Test Eval"):
        images, targets = images.to(device), targets.to(device)
        logits = erm_model(images)
        probs  = torch.softmax(logits, dim=1)
        preds  = torch.argmax(logits, dim=1)

        test_correct += (preds == targets).sum().item()
        test_total   += targets.size(0)

        for i in range(len(targets)):
            g = groups[i].item()
            group_total[g]   = group_total.get(g, 0) + 1
            group_correct[g] = group_correct.get(g, 0) + (preds[i] == targets[i]).item()
            logs.append({
                "image_id":      img_ids[i],
                "label":         targets[i].item(),
                "group":         g,
                "prediction":    preds[i].item(),
                "confidence":    probs[i][preds[i]].item(),
                "logit_class_0": logits[i][0].item(),
                "logit_class_1": logits[i][1].item(),
            })

group_accs = {g: group_correct[g] / (group_total[g] + 1e-8) for g in group_total}
test_wga   = min(group_accs.values())
test_avg   = test_correct / test_total

print("\n=== HAM10000 ERM TEST RESULTS ===")
print(f"Avg Accuracy : {test_avg:.4f}")
print(f"WGA          : {test_wga:.4f}")
for g in sorted(group_accs):
    labels = {0:"non-mel+non-histo", 1:"non-mel+histo", 2:"mel+non-histo(empty)", 3:"mel+histo"}
    print(f"  Group {g} ({labels[g]}): {group_accs[g]:.4f}  n={group_total[g]}")

pd.DataFrame(logs).to_csv("/kaggle/working/ham_erm_test_predictions.csv", index=False)
print("\nSaved: ham_erm_test_predictions.csv")

ERM Test Eval: 100%|██████████| 47/47 [00:08<00:00,  5.61it/s]


=== HAM10000 ERM TEST RESULTS ===
Avg Accuracy : 0.9075
WGA          : 0.6587
  Group 0 (non-mel+non-histo): 0.9843  n=702
  Group 1 (non-mel+histo): 0.8880  n=634
  Group 3 (mel+histo): 0.6587  n=167

Saved: ham_erm_test_predictions.csv


In [12]:
print("\n--- Free Lunch: 95/5 Split ---")

total_train  = len(train_data)
split_95_len = int(0.95 * total_train)
split_5_len  = total_train - split_95_len

train_95_ds, train_5_ds = random_split(
    train_data,
    [split_95_len, split_5_len],
    generator=torch.Generator().manual_seed(42)
)

print(f"Phase 1 backbone set : {split_95_len} samples")
print(f"Phase 2 head set     : {split_5_len} samples")

# Check class and group distribution in the 5% set
targets_5 = [train_data[i][1] for i in train_5_ds.indices]
groups_5  = [train_data[i][2] for i in train_5_ds.indices]
print(f"\n5% split class distribution : {Counter(targets_5)}")
print(f"5% split group distribution : {Counter(groups_5)}")
# Class 0 (non-melanoma) will dominate — sampler fixes this
# Group 3 (mel+histo) should have ~20–40 samples

train_95_loader = DataLoader(
    train_95_ds, batch_size=32, shuffle=True, num_workers=2, pin_memory=True
)

fl_model     = make_resnet50()
optimizer_95 = optim.SGD(fl_model.parameters(), lr=1e-3, momentum=0.9, weight_decay=1e-4)

print(f"\n[Phase 1] Training backbone on {split_95_len} samples for {EPOCHS} epochs")


--- Free Lunch: 95/5 Split ---
Phase 1 backbone set : 6659 samples
Phase 2 head set     : 351 samples

5% split class distribution : Counter({0: 319, 1: 32})
5% split group distribution : Counter({0: 169, 1: 150, 3: 32})
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 191MB/s] 



[Phase 1] Training backbone on 6659 samples for 25 epochs


In [11]:
for epoch in range(EPOCHS):
    fl_model.train()
    total_loss, correct, total = 0.0, 0, 0
    loop = tqdm(train_95_loader, desc=f"FL Backbone Epoch [{epoch+1}/{EPOCHS}]")

    for images, targets, _, _ in loop:
        images, targets = images.to(device), targets.to(device)
        optimizer_95.zero_grad()
        logits = fl_model(images)
        loss   = criterion(logits, targets)
        loss.backward()
        optimizer_95.step()

        total_loss += loss.item()
        preds       = torch.argmax(logits, dim=1)
        correct    += (preds == targets).sum().item()
        total      += targets.size(0)
        loop.set_postfix({"loss": total_loss / (loop.n + 1), "acc": correct / total})

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_95_loader):.4f} | Acc: {correct/total:.4f}")

torch.save(fl_model.state_dict(), "/kaggle/working/ham_fl_backbone.pth")
print("Phase 1 complete. Backbone saved.")

FL Backbone Epoch [1/25]: 100%|██████████| 209/209 [01:02<00:00,  3.32it/s, loss=0.274, acc=0.893]


Epoch 1/25 | Loss: 0.2738 | Acc: 0.8931


FL Backbone Epoch [2/25]: 100%|██████████| 209/209 [01:04<00:00,  3.23it/s, loss=0.218, acc=0.91] 


Epoch 2/25 | Loss: 0.2184 | Acc: 0.9100


FL Backbone Epoch [3/25]: 100%|██████████| 209/209 [01:05<00:00,  3.17it/s, loss=0.187, acc=0.928]


...

Epoch 23/25 | Loss: 0.0353 | Acc: 0.9905


FL Backbone Epoch [24/25]: 100%|██████████| 209/209 [01:07<00:00,  3.10it/s, loss=0.0462, acc=0.982]


Epoch 24/25 | Loss: 0.0462 | Acc: 0.9817


FL Backbone Epoch [25/25]: 100%|██████████| 209/209 [01:07<00:00,  3.11it/s, loss=0.0195, acc=0.994]


Epoch 25/25 | Loss: 0.0195 | Acc: 0.9940
Phase 1 complete. Backbone saved.


In [14]:
fl_model.load_state_dict(torch.load("/kaggle/input/models/ankitaanand26/ham-fl-backbone/pytorch/default/1/ham_fl_backbone.pth"))

<All keys matched successfully>

In [16]:
print("\n[Phase 2] Freezing backbone, retraining classifier head on class-balanced 5% split")

for param in fl_model.parameters():
    param.requires_grad = False

in_features = fl_model.fc.in_features
fl_model.fc = nn.Linear(in_features, 2).to(device)
optimizer_fc = optim.SGD(fl_model.fc.parameters(), lr=1e-2, momentum=0.9, weight_decay=1e-4)

# WeightedRandomSampler — same approach as CelebA, all samples kept
targets_5_arr  = np.array(targets_5)
class_counts   = np.bincount(targets_5_arr)
class_weights  = 1.0 / class_counts
sample_weights = np.array([class_weights[y] for y in targets_5_arr])

sampler = WeightedRandomSampler(
    weights=torch.from_numpy(sample_weights).float(),
    num_samples=len(sample_weights),
    replacement=True
)

train_5_loader = DataLoader(
    train_5_ds, batch_size=32, sampler=sampler, num_workers=2, pin_memory=True
)

print(f"Class counts in 5% set — Class 0: {class_counts[0]}, Class 1: {class_counts[1]}")
print(f"Total head samples: {len(train_5_ds)} (all kept, sampler balances batches)")

HEAD_EPOCHS = 50
fl_model.train()

for epoch in range(HEAD_EPOCHS):
    correct, total, total_loss = 0, 0, 0.0
    for images, targets, _, _ in train_5_loader:
        images, targets = images.to(device), targets.to(device)
        optimizer_fc.zero_grad()
        logits = fl_model(images)
        loss   = criterion(logits, targets)
        loss.backward()
        optimizer_fc.step()
        total_loss += loss.item()
        preds       = torch.argmax(logits, dim=1)
        correct    += (preds == targets).sum().item()
        total      += targets.size(0)

    if (epoch + 1) % 1 == 0:
        print(f"Head epoch {epoch+1}/{HEAD_EPOCHS} | Loss: {total_loss/len(train_5_loader):.4f} | Acc: {correct/total:.4f}")

torch.save(fl_model.state_dict(), "/kaggle/working/ham_fl_final_456.pth")
print("Phase 2 complete. FL model saved.")


[Phase 2] Freezing backbone, retraining classifier head on class-balanced 5% split
Class counts in 5% set — Class 0: 319, Class 1: 32
Total head samples: 351 (all kept, sampler balances batches)
Head epoch 1/50 | Loss: 0.5787 | Acc: 0.7009
Head epoch 2/50 | Loss: 0.4624 | Acc: 0.8262
Head epoch 3/50 | Loss: 0.3152 | Acc: 0.8974
Head epoch 4/50 | Loss: 0.2597 | Acc: 0.9060
Head epoch 5/50 | Loss: 0.2896 | Acc: 0.8946
Head epoch 6/50 | Loss: 0.3944 | Acc: 0.8746
Head epoch 7/50 | Loss: 0.5843 | Acc: 0.8034
Head epoch 8/50 | Loss: 0.9489 | Acc: 0.7863
Head epoch 9/50 | Loss: 0.5832 | Acc: 0.8575
Head epoch 10/50 | Loss: 0.5659 | Acc: 0.8291
Head epoch 11/50 | Loss: 0.2238 | Acc: 0.9231
Head epoch 12/50 | Loss: 0.2847 | Acc: 0.9174
Head epoch 13/50 | Loss: 0.2008 | Acc: 0.9288
Head epoch 14/50 | Loss: 0.0736 | Acc: 0.9772
Head epoch 15/50 | Loss: 0.1540 | Acc: 0.9487
Head epoch 16/50 | Loss: 0.2242 | Acc: 0.9259
Head epoch 17/50 | Loss: 0.4238 | Acc: 0.8632
Head epoch 18/50 | Loss: 0.3594

In [17]:
fl_model.eval()

group_correct, group_total = {}, {}
test_correct, test_total = 0, 0
fl_logs = []

with torch.no_grad():
    for images, targets, groups, img_ids in tqdm(test_loader, desc="FL Test Eval"):
        images, targets = images.to(device), targets.to(device)
        logits = fl_model(images)
        probs  = torch.softmax(logits, dim=1)
        preds  = torch.argmax(logits, dim=1)

        test_correct += (preds == targets).sum().item()
        test_total   += targets.size(0)

        for i in range(len(targets)):
            g = groups[i].item()
            group_total[g]   = group_total.get(g, 0) + 1
            group_correct[g] = group_correct.get(g, 0) + (preds[i] == targets[i]).item()
            fl_logs.append({
                "image_id":      img_ids[i],
                "label":         targets[i].item(),
                "group":         g,
                "fl_prediction": preds[i].item(),
                "fl_confidence": probs[i][preds[i]].item(),
                "fl_logit_0":    logits[i][0].item(),
                "fl_logit_1":    logits[i][1].item(),
            })

group_accs = {g: group_correct[g] / (group_total[g] + 1e-8) for g in group_total}
fl_wga     = min(group_accs.values())
fl_avg     = test_correct / test_total

print("\n=== HAM10000 FL TEST RESULTS ===")
print(f"Avg Accuracy : {fl_avg:.4f}")
print(f"WGA          : {fl_wga:.4f}  ← should be higher than ERM WGA")
for g in sorted(group_accs):
    print(f"  Group {g}: {group_accs[g]:.4f}  n={group_total[g]}")

pd.DataFrame(fl_logs).to_csv("/kaggle/working/ham_fl_test_predictions_456.csv", index=False)
print("\nSaved: ham_fl_test_predictions.csv")

FL Test Eval: 100%|██████████| 47/47 [00:07<00:00,  5.93it/s]


=== HAM10000 FL TEST RESULTS ===
Avg Accuracy : 0.8776
WGA          : 0.7006  ← should be higher than ERM WGA
  Group 0: 0.9715  n=702
  Group 1: 0.8202  n=634
  Group 3: 0.7006  n=167

Saved: ham_fl_test_predictions.csv


In [14]:
erm = pd.read_csv("/kaggle/working/ham_erm_test_predictions.csv")
fl  = pd.read_csv("/kaggle/working/ham_fl_test_predictions.csv")

print(f"ERM rows: {len(erm)} | FL rows: {len(fl)}")  # both should be ~1503

df = erm.merge(fl, on="image_id")
df["disagree"]    = df["prediction"] != df["fl_prediction"]
df["erm_correct"] = df["prediction"] == df["label_x"]

print(f"\nDisagreement rate : {df['disagree'].mean():.4f}  ← expect 5–20%")
print(f"Error|Agreement   : {(~df[df['disagree']==False]['erm_correct']).mean():.4f}")
print(f"Error|Disagreement: {(~df[df['disagree']==True]['erm_correct']).mean():.4f}")
print(f"Error|Disagreement should be substantially higher than Error|Agreement")
print("\n✓ Download both CSVs and both .pth files before session ends")

ERM rows: 1503 | FL rows: 1503

Disagreement rate : 0.0965  ← expect 5–20%
Error|Agreement   : 0.0508
Error|Disagreement: 0.4828
Error|Disagreement should be substantially higher than Error|Agreement

✓ Download both CSVs and both .pth files before session ends


Sanity

In [17]:
set_seed(42)

In [18]:
# 1. Create model
fl_model = models.resnet50(weights=None)
fl_model.fc = nn.Linear(fl_model.fc.in_features, 2)
fl_model = fl_model.to(device)

# 2. Load Phase 1 weights
fl_model.load_state_dict(torch.load("/kaggle/input/models/ankitaanand26/ham-fl-backbone/pytorch/default/1/ham_fl_backbone.pth", map_location=device))

# 3. Freeze backbone
for param in fl_model.parameters():
    param.requires_grad = False

# 4. Replace head
in_features = fl_model.fc.in_features
fl_model.fc = nn.Linear(in_features, 2).to(device)

In [19]:
print("\n[Phase 2] Freezing backbone, retraining classifier head on class-balanced 5% split")

for param in fl_model.parameters():
    param.requires_grad = False

in_features = fl_model.fc.in_features
fl_model.fc = nn.Linear(in_features, 2).to(device)
optimizer_fc = optim.SGD(fl_model.fc.parameters(), lr=1e-2, momentum=0.9, weight_decay=1e-4)

# WeightedRandomSampler — same approach as CelebA, all samples kept
targets_5_arr  = np.array(targets_5)
class_counts   = np.bincount(targets_5_arr)
class_weights  = 1.0 / class_counts
sample_weights = np.array([class_weights[y] for y in targets_5_arr])

sampler = WeightedRandomSampler(
    weights=torch.from_numpy(sample_weights).float(),
    num_samples=len(sample_weights),
    replacement=True
)

train_5_loader = DataLoader(
    train_5_ds, batch_size=32, sampler=sampler, num_workers=2, pin_memory=True
)

print(f"Class counts in 5% set — Class 0: {class_counts[0]}, Class 1: {class_counts[1]}")
print(f"Total head samples: {len(train_5_ds)} (all kept, sampler balances batches)")

HEAD_EPOCHS = 50
fl_model.train()

for epoch in range(HEAD_EPOCHS):
    correct, total, total_loss = 0, 0, 0.0
    for images, targets, _, _ in train_5_loader:
        images, targets = images.to(device), targets.to(device)
        optimizer_fc.zero_grad()
        logits = fl_model(images)
        loss   = criterion(logits, targets)
        loss.backward()
        optimizer_fc.step()
        total_loss += loss.item()
        preds       = torch.argmax(logits, dim=1)
        correct    += (preds == targets).sum().item()
        total      += targets.size(0)

    if (epoch + 1) % 10 == 0:
        print(f"Head epoch {epoch+1}/{HEAD_EPOCHS} | Loss: {total_loss/len(train_5_loader):.4f} | Acc: {correct/total:.4f}")

torch.save(fl_model.state_dict(), "/kaggle/working/ham_fl_final.pth")
print("Phase 2 complete. FL model saved.")


[Phase 2] Freezing backbone, retraining classifier head on class-balanced 5% split
Class counts in 5% set — Class 0: 319, Class 1: 32
Total head samples: 351 (all kept, sampler balances batches)
Head epoch 10/50 | Loss: 0.1185 | Acc: 0.9573
Head epoch 20/50 | Loss: 0.0849 | Acc: 0.9601
Head epoch 30/50 | Loss: 0.0823 | Acc: 0.9687
Head epoch 40/50 | Loss: 0.0682 | Acc: 0.9715
Head epoch 50/50 | Loss: 0.1375 | Acc: 0.9573
Phase 2 complete. FL model saved.


In [20]:
fl_model.eval()

group_correct, group_total = {}, {}
test_correct, test_total = 0, 0
fl_logs = []

with torch.no_grad():
    for images, targets, groups, img_ids in tqdm(test_loader, desc="FL Test Eval"):
        images, targets = images.to(device), targets.to(device)
        logits = fl_model(images)
        probs  = torch.softmax(logits, dim=1)
        preds  = torch.argmax(logits, dim=1)

        test_correct += (preds == targets).sum().item()
        test_total   += targets.size(0)

        for i in range(len(targets)):
            g = groups[i].item()
            group_total[g]   = group_total.get(g, 0) + 1
            group_correct[g] = group_correct.get(g, 0) + (preds[i] == targets[i]).item()
            fl_logs.append({
                "image_id":      img_ids[i],
                "label":         targets[i].item(),
                "group":         g,
                "fl_prediction": preds[i].item(),
                "fl_confidence": probs[i][preds[i]].item(),
                "fl_logit_0":    logits[i][0].item(),
                "fl_logit_1":    logits[i][1].item(),
            })

group_accs = {g: group_correct[g] / (group_total[g] + 1e-8) for g in group_total}
fl_wga     = min(group_accs.values())
fl_avg     = test_correct / test_total

print("\n=== HAM10000 FL TEST RESULTS ===")
print(f"Avg Accuracy : {fl_avg:.4f}")
print(f"WGA          : {fl_wga:.4f}  ← should be higher than ERM WGA")
for g in sorted(group_accs):
    print(f"  Group {g}: {group_accs[g]:.4f}  n={group_total[g]}")

pd.DataFrame(fl_logs).to_csv("/kaggle/working/ham_fl_test_predictions.csv", index=False)
print("\nSaved: ham_fl_test_predictions.csv")

FL Test Eval: 100%|██████████| 47/47 [00:07<00:00,  5.99it/s]


=== HAM10000 FL TEST RESULTS ===
Avg Accuracy : 0.9255
WGA          : 0.5150  ← should be higher than ERM WGA
  Group 0: 0.9929  n=702
  Group 1: 0.9590  n=634
  Group 3: 0.5150  n=167

Saved: ham_fl_test_predictions.csv


In [22]:
erm = pd.read_csv("/kaggle/input/datasets/ankitaanand26/ham-erm-test-predictions/ham_erm_test_predictions (1).csv")
fl  = pd.read_csv("/kaggle/working/ham_fl_test_predictions.csv")

print(f"ERM rows: {len(erm)} | FL rows: {len(fl)}")  # both should be ~1503

df = erm.merge(fl, on="image_id")
df["disagree"]    = df["prediction"] != df["fl_prediction"]
df["erm_correct"] = df["prediction"] == df["label_x"]

print(f"\nDisagreement rate : {df['disagree'].mean():.4f}  ← expect 5–20%")
print(f"Error|Agreement   : {(~df[df['disagree']==False]['erm_correct']).mean():.4f}")
print(f"Error|Disagreement: {(~df[df['disagree']==True]['erm_correct']).mean():.4f}")
print(f"Error|Disagreement should be substantially higher than Error|Agreement")
print("\n✓ Download both CSVs and both .pth files before session ends")

ERM rows: 1503 | FL rows: 1503

Disagreement rate : 0.0845  ← expect 5–20%
Error|Agreement   : 0.0451
Error|Disagreement: 0.6063
Error|Disagreement should be substantially higher than Error|Agreement

✓ Download both CSVs and both .pth files before session ends


Seed 123 ERM

In [10]:
erm_model = make_resnet50()
optimizer = optim.SGD(erm_model.parameters(), lr=1e-3, momentum=0.9, weight_decay=1e-4)

print("Starting HAM10000 ERM Training...")
best_val_wga = 0.0

for epoch in range(EPOCHS):
    erm_model.train()
    train_loss, correct, total = 0.0, 0, 0
    loop = tqdm(train_loader, desc=f"ERM Epoch [{epoch+1}/{EPOCHS}]")

    for images, targets, groups, _ in loop:
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        logits = erm_model(images)
        loss   = criterion(logits, targets)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        preds       = torch.argmax(logits, dim=1)
        correct    += (preds == targets).sum().item()
        total      += targets.size(0)
        loop.set_postfix({"loss": train_loss / (loop.n + 1), "acc": correct / total})

    # Validation
    erm_model.eval()
    group_correct = {}
    group_total   = {}
    val_correct, val_total = 0, 0

    with torch.no_grad():
        for images, targets, groups, _ in val_loader:
            images, targets = images.to(device), targets.to(device)
            logits = erm_model(images)
            preds  = torch.argmax(logits, dim=1)
            val_correct += (preds == targets).sum().item()
            val_total   += targets.size(0)
            for i in range(len(targets)):
                g = groups[i].item()
                group_total[g]   = group_total.get(g, 0) + 1
                group_correct[g] = group_correct.get(g, 0) + (preds[i] == targets[i]).item()

    group_accs = {g: group_correct[g] / (group_total[g] + 1e-8) for g in group_total}
    wga     = min(group_accs.values())
    avg_acc = val_correct / val_total

    group_str = "  ".join([f"G{g}:{group_accs[g]:.3f}" for g in sorted(group_accs)])
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Train Acc: {correct/total:.4f} "
          f"| Val Acc: {avg_acc:.4f} | Val WGA: {wga:.4f} | {group_str}")

    if wga > best_val_wga:
        best_val_wga = wga
        torch.save(erm_model.state_dict(), "/kaggle/working/ham_erm_best.pth")
        print(f"  → New best WGA: {wga:.4f} — checkpoint saved")

torch.save(erm_model.state_dict(), "/kaggle/working/ham_erm_final.pth")
print(f"\nERM training complete. Best val WGA: {best_val_wga:.4f}")

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 191MB/s]


Starting HAM10000 ERM Training...


ERM Epoch [1/25]: 100%|██████████| 220/220 [01:07<00:00,  3.26it/s, loss=0.283, acc=0.893]


Epoch 01/25 | Train Acc: 0.8926 | Val Acc: 0.9134 | Val WGA: 0.2874 | G0:0.996  G1:0.987  G3:0.287
  → New best WGA: 0.2874 — checkpoint saved


...

Epoch 23/25 | Train Acc: 0.9756 | Val Acc: 0.9328 | Val WGA: 0.5090 | G0:0.991  G1:0.979  G3:0.509


ERM Epoch [24/25]: 100%|██████████| 220/220 [01:22<00:00,  2.68it/s, loss=0.0212, acc=0.993]


Epoch 24/25 | Train Acc: 0.9933 | Val Acc: 0.9354 | Val WGA: 0.5389 | G0:0.994  G1:0.975  G3:0.539


ERM Epoch [25/25]: 100%|██████████| 220/220 [01:21<00:00,  2.71it/s, loss=0.0177, acc=0.994]


Epoch 25/25 | Train Acc: 0.9944 | Val Acc: 0.9301 | Val WGA: 0.4850 | G0:0.996  G1:0.975  G3:0.485

ERM training complete. Best val WGA: 0.7665


In [11]:
erm_model.load_state_dict(torch.load("/kaggle/working/ham_erm_best.pth"))
erm_model.eval()

group_correct, group_total = {}, {}
test_correct, test_total = 0, 0
logs = []

with torch.no_grad():
    for images, targets, groups, img_ids in tqdm(test_loader, desc="ERM Test Eval"):
        images, targets = images.to(device), targets.to(device)
        logits = erm_model(images)
        probs  = torch.softmax(logits, dim=1)
        preds  = torch.argmax(logits, dim=1)

        test_correct += (preds == targets).sum().item()
        test_total   += targets.size(0)

        for i in range(len(targets)):
            g = groups[i].item()
            group_total[g]   = group_total.get(g, 0) + 1
            group_correct[g] = group_correct.get(g, 0) + (preds[i] == targets[i]).item()
            logs.append({
                "image_id":      img_ids[i],
                "label":         targets[i].item(),
                "group":         g,
                "prediction":    preds[i].item(),
                "confidence":    probs[i][preds[i]].item(),
                "logit_class_0": logits[i][0].item(),
                "logit_class_1": logits[i][1].item(),
            })

group_accs = {g: group_correct[g] / (group_total[g] + 1e-8) for g in group_total}
test_wga   = min(group_accs.values())
test_avg   = test_correct / test_total

print("\n=== HAM10000 ERM TEST RESULTS ===")
print(f"Avg Accuracy : {test_avg:.4f}")
print(f"WGA          : {test_wga:.4f}")
for g in sorted(group_accs):
    labels = {0:"non-mel+non-histo", 1:"non-mel+histo", 2:"mel+non-histo(empty)", 3:"mel+histo"}
    print(f"  Group {g} ({labels[g]}): {group_accs[g]:.4f}  n={group_total[g]}")

pd.DataFrame(logs).to_csv("/kaggle/working/ham_erm_test_predictions.csv", index=False)
print("\nSaved: ham_erm_test_predictions.csv")

ERM Test Eval: 100%|██████████| 47/47 [00:07<00:00,  6.11it/s]


=== HAM10000 ERM TEST RESULTS ===
Avg Accuracy : 0.9095
WGA          : 0.6587
  Group 0 (non-mel+non-histo): 0.9843  n=702
  Group 1 (non-mel+histo): 0.8927  n=634
  Group 3 (mel+histo): 0.6587  n=167

Saved: ham_erm_test_predictions.csv


Seed 456 ERM

In [9]:
erm_model = make_resnet50()
optimizer = optim.SGD(erm_model.parameters(), lr=1e-3, momentum=0.9, weight_decay=1e-4)

print("Starting HAM10000 ERM Training...")
best_val_wga = 0.0

for epoch in range(EPOCHS):
    erm_model.train()
    train_loss, correct, total = 0.0, 0, 0
    loop = tqdm(train_loader, desc=f"ERM Epoch [{epoch+1}/{EPOCHS}]")

    for images, targets, groups, _ in loop:
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        logits = erm_model(images)
        loss   = criterion(logits, targets)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        preds       = torch.argmax(logits, dim=1)
        correct    += (preds == targets).sum().item()
        total      += targets.size(0)
        loop.set_postfix({"loss": train_loss / (loop.n + 1), "acc": correct / total})

    # Validation
    erm_model.eval()
    group_correct = {}
    group_total   = {}
    val_correct, val_total = 0, 0

    with torch.no_grad():
        for images, targets, groups, _ in val_loader:
            images, targets = images.to(device), targets.to(device)
            logits = erm_model(images)
            preds  = torch.argmax(logits, dim=1)
            val_correct += (preds == targets).sum().item()
            val_total   += targets.size(0)
            for i in range(len(targets)):
                g = groups[i].item()
                group_total[g]   = group_total.get(g, 0) + 1
                group_correct[g] = group_correct.get(g, 0) + (preds[i] == targets[i]).item()

    group_accs = {g: group_correct[g] / (group_total[g] + 1e-8) for g in group_total}
    wga     = min(group_accs.values())
    avg_acc = val_correct / val_total

    group_str = "  ".join([f"G{g}:{group_accs[g]:.3f}" for g in sorted(group_accs)])
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Train Acc: {correct/total:.4f} "
          f"| Val Acc: {avg_acc:.4f} | Val WGA: {wga:.4f} | {group_str}")

    if wga > best_val_wga:
        best_val_wga = wga
        torch.save(erm_model.state_dict(), "/kaggle/working/ham_erm_best_seed456.pth")
        print(f"  → New best WGA: {wga:.4f} — checkpoint saved")

torch.save(erm_model.state_dict(), "/kaggle/working/ham_erm_final_seed456.pth")
print(f"\nERM training complete. Best val WGA: {best_val_wga:.4f}")

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 188MB/s]


Starting HAM10000 ERM Training...


ERM Epoch [1/25]: 100%|██████████| 220/220 [01:09<00:00,  3.16it/s, loss=0.277, acc=0.893]


Epoch 01/25 | Train Acc: 0.8932 | Val Acc: 0.9088 | Val WGA: 0.3353 | G0:0.989  G1:0.972  G3:0.335
  → New best WGA: 0.3353 — checkpoint saved


...

Epoch 23/25 | Train Acc: 0.9977 | Val Acc: 0.9401 | Val WGA: 0.6707 | G0:0.994  G1:0.951  G3:0.671


ERM Epoch [24/25]: 100%|██████████| 220/220 [01:11<00:00,  3.06it/s, loss=0.0101, acc=0.997] 


Epoch 24/25 | Train Acc: 0.9970 | Val Acc: 0.9261 | Val WGA: 0.6707 | G0:0.990  G1:0.923  G3:0.671


ERM Epoch [25/25]: 100%|██████████| 220/220 [01:11<00:00,  3.06it/s, loss=0.0196, acc=0.993]


Epoch 25/25 | Train Acc: 0.9927 | Val Acc: 0.9434 | Val WGA: 0.6287 | G0:0.997  G1:0.967  G3:0.629

ERM training complete. Best val WGA: 0.6886


In [10]:
erm_model.load_state_dict(torch.load("/kaggle/working/ham_erm_best_seed456.pth"))
erm_model.eval()

group_correct, group_total = {}, {}
test_correct, test_total = 0, 0
logs = []

with torch.no_grad():
    for images, targets, groups, img_ids in tqdm(test_loader, desc="ERM Test Eval"):
        images, targets = images.to(device), targets.to(device)
        logits = erm_model(images)
        probs  = torch.softmax(logits, dim=1)
        preds  = torch.argmax(logits, dim=1)

        test_correct += (preds == targets).sum().item()
        test_total   += targets.size(0)

        for i in range(len(targets)):
            g = groups[i].item()
            group_total[g]   = group_total.get(g, 0) + 1
            group_correct[g] = group_correct.get(g, 0) + (preds[i] == targets[i]).item()
            logs.append({
                "image_id":      img_ids[i],
                "label":         targets[i].item(),
                "group":         g,
                "prediction":    preds[i].item(),
                "confidence":    probs[i][preds[i]].item(),
                "logit_class_0": logits[i][0].item(),
                "logit_class_1": logits[i][1].item(),
            })

group_accs = {g: group_correct[g] / (group_total[g] + 1e-8) for g in group_total}
test_wga   = min(group_accs.values())
test_avg   = test_correct / test_total

print("\n=== HAM10000 ERM TEST RESULTS ===")
print(f"Avg Accuracy : {test_avg:.4f}")
print(f"WGA          : {test_wga:.4f}")
for g in sorted(group_accs):
    labels = {0:"non-mel+non-histo", 1:"non-mel+histo", 2:"mel+non-histo(empty)", 3:"mel+histo"}
    print(f"  Group {g} ({labels[g]}): {group_accs[g]:.4f}  n={group_total[g]}")

pd.DataFrame(logs).to_csv("/kaggle/working/ham_erm_test_predictions_seed456.csv", index=False)
print("\nSaved: ham_erm_test_predictions_seed456.csv")

ERM Test Eval: 100%|██████████| 47/47 [00:08<00:00,  5.30it/s]


=== HAM10000 ERM TEST RESULTS ===
Avg Accuracy : 0.9355
WGA          : 0.6587
  Group 0 (non-mel+non-histo): 0.9929  n=702
  Group 1 (non-mel+histo): 0.9448  n=634
  Group 3 (mel+histo): 0.6587  n=167

Saved: ham_erm_test_predictions_seed456.csv
